# 03 · Data Warehouse (R3)
Parte leyendo la capa limpia que deja `02_calidad` (`data/clean/covid_diario.parquet`) y guarda el DW en
`data/dw/covid_dw.db` (SQLite). Sigue los pasos del taller de la S10: dimensión de fecha, dimensiones con
llave sustituta, tabla de hechos y conciliación.

| Historia | Sección |
|---|---|
| H3.1 · Estrella en SQLite | 1 a 4 |
| H3.2 · Fuente externa: vacunación de OWID | 5 |
| H3.3 · Fuente externa 2: Banco Mundial | 6 |
| H3.4 · Consultas OLAP | 7 |

Dos tablas de hechos que **comparten las dimensiones** (dimensiones conformadas, S9):
- `fact_covid`: **una fila por territorio × día** (territorio = país + región/estado + condado, tal como lo informa JHU ese día).
- `fact_vacunacion`: **una fila por país × día** (Our World in Data).

```
     fact_covid                                              fact_vacunacion
 (acumulados, nuevos,          dim_fecha                    (% 1 dosis, % esquema
  marcas de calidad)    ──  (fecha_key, año, mes…)  ──       completo, % refuerzo)
          \                                                      /
           \──────────────── dim_territorio ───────────────────/
                 (territorio_key, país → región/estado → condado, población, iso3)
```

In [1]:
import os
import sqlite3
import pandas as pd

df = pd.read_parquet("../data/clean/covid_diario.parquet")
print("Filas:", len(df), "| columnas:", df.shape[1])

Filas: 4276477 | columnas: 26


## 1 · Dimensión de fecha
No viene de ningún archivo: se genera con una fila por cada día del período, del 22-ene-2020 al 9-mar-2023
(1.143 días). Se incluyen todos los días aunque algún país no informe: así las consultas pueden mostrar el hueco.

- `fecha_key` es la fecha como número AAAAMMDD (20200122), igual que en la S10.
- `inicio_semana` es el lunes de la semana: sirve para agrupar por semana sin el problema de la semana 53
  que cruza de un año a otro.

In [2]:
f = pd.date_range("2020-01-22", "2023-03-09", freq="D")

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
         "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

dim_fecha = pd.DataFrame({
    "fecha_key":     f.strftime("%Y%m%d").astype(int),
    "fecha":         f.strftime("%Y-%m-%d"),
    "anio":          f.year,
    "trimestre":     f.quarter,
    "mes":           f.month,
    "nombre_mes":    [MESES[m - 1] for m in f.month],
    "inicio_semana": (f - pd.to_timedelta(f.day_of_week, unit="D")).strftime("%Y-%m-%d"),
    "dia_semana":    [DIAS[d] for d in f.day_of_week],
    "es_fin_semana": (f.day_of_week >= 5).astype(int),
})
print("dim_fecha:", len(dim_fecha), "días | desde", dim_fecha["fecha"].iloc[0], "hasta", dim_fecha["fecha"].iloc[-1])
dim_fecha.head()

dim_fecha: 1143 días | desde 2020-01-22 hasta 2023-03-09


,fecha_key,fecha,anio,trimestre,mes,nombre_mes,inicio_semana,dia_semana,es_fin_semana
0,20200122,2020-01-22,2020,1,1,enero,2020-01-20,miércoles,0
1,20200123,2020-01-23,2020,1,1,enero,2020-01-20,jueves,0
2,20200124,2020-01-24,2020,1,1,enero,2020-01-20,viernes,0
3,20200125,2020-01-25,2020,1,1,enero,2020-01-20,sábado,1
4,20200126,2020-01-26,2020,1,1,enero,2020-01-20,domingo,1


## 2 · Dimensión de territorio
Una fila por cada territorio que aparece en los datos (país + región/estado + condado), con una **llave sustituta**
`territorio_key` (como `paciente_key` en la S10). El `uid` de JHU queda como un atributo más: así también entran los
territorios que no cruzan con la tabla UID (ciudades de EE.UU. y Canadá de feb–mar 2020, ver H1.3).

- **Jerarquía:** `pais` → `region_estado` → `condado`. Un texto vacío significa "el territorio completo".
- `nivel`: `pais`, `region` o `condado`, según la columna más fina que tenga dato.
- De la tabla UID (leída desde raw) se traen `iso3`, `poblacion`, `lat` y `long` del territorio.
- `poblacion_pais`: población del país completo (fila de UID sin provincia). Hace falta para las tasas por 100.000
  al resumir por país: sumar la población de los territorios de un día no sirve, porque Chile pasa de 1 fila
  nacional a 17 regionales el 20-may-2020.
- **Una fila de país para cada país.** Canadá, China, Malasia y EE.UU. llegan solo por región o estado; se les agrega
  su fila de país (con `uid` e `iso3` de UID). No tiene hechos en `fact_covid`, pero la usa `fact_vacunacion` (sección 5),
  que es por país.

In [3]:
URL_UID = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"
uid = pd.read_csv(URL_UID)

llave = ["country_region", "province_state", "admin2"]

fila_pais = uid[uid["Province_State"].isna()].set_index("Country_Region")   # fila del país completo en UID
poblacion_pais = fila_pais["Population"]

terr = df[llave + ["uid"]].drop_duplicates(llave)

# Países que JHU informa solo por región/estado: se les agrega su fila de país (sin hechos en fact_covid)
con_fila_pais = terr.loc[(terr["province_state"] == "") & (terr["admin2"] == ""), "country_region"]
sin_fila_pais = sorted(set(terr["country_region"]) - set(con_fila_pais))
print("Países sin fila de país en los reportes:", sin_fila_pais)
nuevas = pd.DataFrame({"country_region": sin_fila_pais, "province_state": "", "admin2": "",
                       "uid": fila_pais.loc[sin_fila_pais, "UID"].values})
terr = pd.concat([terr, nuevas]).sort_values(llave).reset_index(drop=True)

terr = terr.merge(uid[["UID", "iso3", "Population", "Lat", "Long_"]], left_on="uid", right_on="UID", how="left")

dim_territorio = pd.DataFrame({
    "territorio_key": range(1, len(terr) + 1),          # llave sustituta
    "uid":            terr["uid"],
    "pais":           terr["country_region"],
    "region_estado":  terr["province_state"],
    "condado":        terr["admin2"],
    "nivel":          "pais",
    "iso3":           terr["iso3"],
    "poblacion":      terr["Population"],
    "poblacion_pais": terr["country_region"].map(poblacion_pais),
    "lat":            terr["Lat"],
    "long":           terr["Long_"],
})
dim_territorio.loc[dim_territorio["region_estado"] != "", "nivel"] = "region"
dim_territorio.loc[dim_territorio["condado"] != "", "nivel"] = "condado"

print("Territorios:", len(dim_territorio), "| países:", dim_territorio["pais"].nunique())
print("Por nivel:", dim_territorio["nivel"].value_counts().to_dict())
print("Sin uid:", dim_territorio["uid"].isna().sum(),
      "| sin población:", dim_territorio["poblacion"].isna().sum(),
      "| países sin población:", dim_territorio.loc[dim_territorio["poblacion_pais"].isna(), "pais"].nunique())
dim_territorio[dim_territorio["pais"] == "Chile"]

Países sin fila de país en los reportes: ['Canada', 'China', 'Malaysia', 'US']
Territorios: 4280 | países: 196
Por nivel: {'condado': 3318, 'region': 766, 'pais': 196}
Sin uid: 172 | sin población: 259 | países sin población: 0


,territorio_key,uid,pais,region_estado,condado,nivel,iso3,poblacion,poblacion_pais,lat,long
104,105,152.0,Chile,,,pais,CHL,19116209.0,19116209.0,-35.6751,-71.5430
105,106,15201.0,Chile,Antofagasta,,region,CHL,607534.0,19116209.0,-23.6509,-70.3975
106,107,15202.0,Chile,Araucania,,region,CHL,957224.0,19116209.0,-38.9489,-72.3311
107,108,15203.0,Chile,Arica y Parinacota,,region,CHL,226068.0,19116209.0,-18.5940,-69.4785
108,109,15204.0,Chile,Atacama,,region,CHL,288944.0,19116209.0,-27.5661,-70.0503
109,110,15205.0,Chile,Aysen,,region,CHL,103158.0,19116209.0,-45.9864,-73.7669
110,111,15206.0,Chile,Biobio,,region,CHL,1556805.0,19116209.0,-37.4464,-72.1416
111,112,15207.0,Chile,Coquimbo,,region,CHL,757586.0,19116209.0,-29.9590,-71.3389
112,113,15208.0,Chile,Los Lagos,,region,CHL,828708.0,19116209.0,-41.9198,-72.1416
113,114,15209.0,Chile,Los Rios,,region,CHL,384837.0,19116209.0,-40.2310,-72.3311


## 3 · Tabla de hechos
Una fila por territorio × día. Tiene solo llaves y números (S10):

- **Acumulados** tal como los publicó JHU (`casos_acumulados`, `fallecidos_acumulados`) y **nuevos del día**
  (`casos_nuevos`, `fallecidos_nuevos`, calculados en H2.1).
- **Marcas de calidad** de `02_calidad` (1 = sí, 0 = no): se cargan para que el análisis y el tablero puedan avisar.
- **Linaje:** `fila_origen`. El archivo de origen no se guarda porque se deduce de la fecha (`MM-DD-AAAA.csv`).

Los nuevos de un **país** no se suman desde esta tabla cuando cambia la granularidad (Chile el 20-may-2020): ese día
quedan vacíos (`primer_dia_por_cambio`). Para el país se suma el acumulado por día y se resta el día anterior
(opción C de H2.1).

In [4]:
fact_covid = df.merge(dim_territorio[["territorio_key", "pais", "region_estado", "condado"]],
                      left_on=llave, right_on=["pais", "region_estado", "condado"])

fact_covid = pd.DataFrame({
    "fecha_key":             fact_covid["fecha_reporte"].dt.strftime("%Y%m%d").astype(int),
    "territorio_key":        fact_covid["territorio_key"],
    "casos_acumulados":      fact_covid["confirmed"],
    "fallecidos_acumulados": fact_covid["deaths"],
    "casos_nuevos":          fact_covid["casos_nuevos"],
    "fallecidos_nuevos":     fact_covid["fallecidos_nuevos"],
    "retrocede_casos":       fact_covid["retrocede_casos"].astype(int),
    "retrocede_fallecidos":  fact_covid["retrocede_fallecidos"].astype(int),
    "primer_dia_por_cambio": fact_covid["primer_dia_por_cambio"].astype(int),
    "sin_reporte":           fact_covid["sin_reporte"].astype(int),
    "pico_rezago":           fact_covid["pico_rezago"].astype(int),
    "fila_origen":           fact_covid["fila_origen"],
})

print("fact_covid:", len(fact_covid), "filas (Parquet:", len(df), ")")
print("Grano territorio × día sin repetidos:", not fact_covid.duplicated(["territorio_key", "fecha_key"]).any())
print("Todas las fechas existen en dim_fecha:", fact_covid["fecha_key"].isin(dim_fecha["fecha_key"]).all())
fact_covid.head()

fact_covid: 4276477 filas (Parquet: 4276477 )


Grano territorio × día sin repetidos: True
Todas las fechas existen en dim_fecha: True


,fecha_key,territorio_key,casos_acumulados,fallecidos_acumulados,casos_nuevos,fallecidos_nuevos,retrocede_casos,retrocede_fallecidos,primer_dia_por_cambio,sin_reporte,pico_rezago,fila_origen
0,20200224,1,5.0,0.0,5.0,0.0,0,0,0,0,0,62
1,20200225,1,5.0,0.0,0.0,0.0,0,0,0,1,0,66
2,20200226,1,5.0,0.0,0.0,0.0,0,0,0,1,0,74
3,20200227,1,5.0,0.0,0.0,0.0,0,0,0,1,0,76
4,20200228,1,5.0,0.0,0.0,0.0,0,0,0,1,0,78


## 4 · Carga y conciliación
Se guardan las tres tablas en `data/dw/covid_dw.db` con `to_sql` (como en la S10) y se comprueba que el DW tiene lo
mismo que el Parquet. **Conciliar** = las mismas cifras en la fuente y en el DW; si alguna fila dice REVISAR, no seguir.

In [5]:
RUTA_DW = "../data/dw/covid_dw.db"
if os.path.exists(RUTA_DW):
    os.remove(RUTA_DW)                     # se crea desde cero en cada corrida

dw = sqlite3.connect(RUTA_DW)
for nombre, tabla in {"dim_fecha": dim_fecha, "dim_territorio": dim_territorio, "fact_covid": fact_covid}.items():
    tabla.to_sql(nombre, dw, if_exists="replace", index=False)
    print(nombre, "→", len(tabla), "filas")

print(f"\nTamaño de {RUTA_DW}: {os.path.getsize(RUTA_DW) / 2**20:.1f} MB")

dim_fecha

 → 1143 filas
dim_territorio → 4280 filas


fact_covid → 4276477 filas

Tamaño de ../data/dw/covid_dw.db: 129.5 MB


In [6]:
chile_final = df[(df["country_region"] == "Chile") & (df["fecha_reporte"] == "2023-03-09")]

CHILE_DW = """
    SELECT SUM(f.casos_acumulados) AS casos, SUM(f.fallecidos_acumulados) AS fallecidos
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile' AND d.fecha = '2023-03-09'
"""
chile_dw = pd.read_sql(CHILE_DW, dw)

control = pd.DataFrame({
    "medida": ["filas", "casos nuevos (mundo)", "fallecidos nuevos (mundo)",
               "Chile · casos acumulados al 9-mar-2023", "Chile · fallecidos acumulados al 9-mar-2023"],
    "Parquet": [len(df), df["casos_nuevos"].sum(), df["fallecidos_nuevos"].sum(),
                chile_final["confirmed"].sum(), chile_final["deaths"].sum()],
    "DW": [pd.read_sql("SELECT COUNT(*) n FROM fact_covid", dw)["n"][0],
           pd.read_sql("SELECT SUM(casos_nuevos) s FROM fact_covid", dw)["s"][0],
           pd.read_sql("SELECT SUM(fallecidos_nuevos) s FROM fact_covid", dw)["s"][0],
           chile_dw["casos"][0], chile_dw["fallecidos"][0]],
})
control["cuadra"] = (control["Parquet"] == control["DW"]).map({True: "OK", False: "REVISAR"})
control

,medida,Parquet,DW,cuadra
0,filas,4276477.0,4276477.0,OK
1,casos nuevos (mundo),676722152.0,676722152.0,OK
2,fallecidos nuevos (mundo),6899598.0,6899598.0,OK
3,Chile · casos acumulados al 9-mar-2023,5192286.0,5192286.0,OK
4,Chile · fallecidos acumulados al 9-mar-2023,64273.0,64273.0,OK


## 5 · Fuente externa: vacunación de Our World in Data (H3.2)
OWID publica, por país y día, el porcentaje de la población vacunada. Se lee desde raw (sin descargar el archivo)
y se cruza con el DW por el **código ISO3** del país (`iso_code` de OWID ↔ `iso3` de la tabla UID).
La usan P3 (letalidad vs % vacunados) y P7 (llegar al 50 % con esquema completo).

- Se cargan solo las fechas hasta el 9-mar-2023, que es lo que cubre `dim_fecha` (OWID sigue hasta 2024).
- Se guardan tres medidas: % con al menos una dosis, % con esquema completo y % con refuerzo (`*_per_hundred`).
- **Los huecos no se rellenan.** Muchos países informan solo algunos días; imputar es inventar.

In [7]:
URL_OWID = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv"
owid = pd.read_csv(URL_OWID)
print("OWID:", len(owid), "filas | desde", owid["date"].min(), "hasta", owid["date"].max())

owid = owid[owid["date"] <= "2023-03-09"]
print("Hasta el 9-mar-2023:", len(owid), "filas")

# Kosovo: UID usa XKS y OWID un código propio. Es la única diferencia de códigos entre ambas fuentes.
EQUIVALENCIAS = {"OWID_KOS": "XKS"}
owid["iso3"] = owid["iso_code"].replace(EQUIVALENCIAS)

OWID: 196246 filas | desde 2020-12-02 hasta 2024-08-14
Hasta el 9-mar-2023: 163154 filas


### Tasa de cruce
Para cada país del DW: ¿está en OWID? ¿trae el % con esquema completo algún día?

In [8]:
paises_dw = dim_territorio.loc[dim_territorio["nivel"] == "pais", ["territorio_key", "pais", "iso3"]]

cruce = paises_dw.copy()
cruce["cruza"] = cruce["iso3"].isin(owid["iso3"])
dias_esquema = owid.groupby("iso3")["people_fully_vaccinated_per_hundred"].count()
cruce["dias_con_esquema_completo"] = cruce["iso3"].map(dias_esquema).fillna(0).astype(int)

print(f"Países que cruzan: {cruce['cruza'].sum()} de {len(cruce)} ({cruce['cruza'].mean():.1%})")
print("Cruzan pero sin ningún dato de esquema completo:",
      cruce.loc[cruce["cruza"] & (cruce["dias_con_esquema_completo"] == 0), "pais"].tolist())

no_cruzan = cruce[~cruce["cruza"]][["pais", "iso3"]].assign(motivo="OWID no publica datos de vacunación de este país")
no_cruzan

Países que cruzan: 190 de 196 (96.9%)
Cruzan pero sin ningún dato de esquema completo: ['Liechtenstein', 'Switzerland']


,pais,iso3,motivo
214,Eritrea,ERI,OWID no publica datos de vacunación de este país
266,Holy See,VAT,OWID no publica datos de vacunación de este país
391,"Korea, North",PRK,OWID no publica datos de vacunación de este país
428,Marshall Islands,MHL,OWID no publica datos de vacunación de este país
465,Micronesia,FSM,OWID no publica datos de vacunación de este país
511,Palau,PLW,OWID no publica datos de vacunación de este país


### Lo que está en OWID y no en el DW
Los códigos que empiezan con `OWID_` son propios de OWID: agregados (mundo, continentes, grupos de ingreso) o partes
de un país (Inglaterra, Escocia…). El resto son territorios que en JHU van como región de otro país, o que no están en
los reportes de JHU.

In [9]:
region_de = dim_territorio[dim_territorio["nivel"] != "pais"].drop_duplicates("iso3").set_index("iso3")["pais"]

otros = owid.loc[~owid["iso3"].isin(paises_dw["iso3"]), ["iso_code", "location"]].drop_duplicates()
otros["motivo"] = "no está en los reportes de JHU"
otros.loc[otros["iso_code"].isin(region_de.index), "motivo"] = "en JHU es una región de " + otros["iso_code"].map(region_de)
otros.loc[otros["iso_code"].str.startswith("OWID_"), "motivo"] = "código propio de OWID (agregado o parte de un país)"

print("Códigos de OWID sin pareja:", len(otros),
      "| propios de OWID:", otros["iso_code"].str.startswith("OWID_").sum(),
      "| regiones de otro país en JHU:", otros["motivo"].str.startswith("en JHU").sum(),
      "| no están en JHU:", (otros["motivo"] == "no está en los reportes de JHU").sum())
otros.sort_values(["motivo", "iso_code"]).reset_index(drop=True)

Códigos de OWID sin pareja:

 45 | propios de OWID: 17 | regiones de otro país en JHU: 26 | no están en JHU: 2


,iso_code,location,motivo
0,OWID_AFR,Africa,código propio de OWID (agregado o parte de un ...
1,OWID_ASI,Asia,código propio de OWID (agregado o parte de un ...
2,OWID_CYN,Northern Cyprus,código propio de OWID (agregado o parte de un ...
3,OWID_ENG,England,código propio de OWID (agregado o parte de un ...
4,OWID_EUN,European Union,código propio de OWID (agregado o parte de un ...
5,OWID_EUR,Europe,código propio de OWID (agregado o parte de un ...
6,OWID_HIC,High income,código propio de OWID (agregado o parte de un ...
7,OWID_LIC,Low income,código propio de OWID (agregado o parte de un ...
8,OWID_LMC,Lower middle income,código propio de OWID (agregado o parte de un ...
9,OWID_NAM,North America,código propio de OWID (agregado o parte de un ...


### Tabla de hechos de vacunación
Una fila por país × día, apuntando a la fila de país de `dim_territorio`. Se descartan las filas de OWID que no traen
ninguno de los tres porcentajes (solo informan dosis aplicadas).

In [10]:
MEDIDAS = {"people_vaccinated_per_hundred": "pct_una_dosis",
           "people_fully_vaccinated_per_hundred": "pct_esquema_completo",
           "total_boosters_per_hundred": "pct_refuerzo"}

v = owid.merge(paises_dw[["territorio_key", "iso3"]], on="iso3").dropna(subset=list(MEDIDAS), how="all")

fact_vacunacion = pd.DataFrame({
    "fecha_key":      pd.to_datetime(v["date"]).dt.strftime("%Y%m%d").astype(int),
    "territorio_key": v["territorio_key"],
    **{nuevo: v[original] for original, nuevo in MEDIDAS.items()},
})
fact_vacunacion.to_sql("fact_vacunacion", dw, if_exists="replace", index=False)

print("fact_vacunacion:", len(fact_vacunacion), "filas | países:", fact_vacunacion["territorio_key"].nunique())
print("Grano país × día sin repetidos:", not fact_vacunacion.duplicated(["territorio_key", "fecha_key"]).any())
print("Todas las fechas existen en dim_fecha:", fact_vacunacion["fecha_key"].isin(dim_fecha["fecha_key"]).all())

fact_vacunacion: 54905 filas | países: 190
Grano país × día sin repetidos: True
Todas las fechas existen en dim_fecha: True


### Conciliación

In [11]:
chile_owid = owid[owid["iso3"] == "CHL"].dropna(subset=["people_fully_vaccinated_per_hundred"]).iloc[-1]

CHILE_VAC = """
    SELECT d.fecha, v.pct_esquema_completo
    FROM fact_vacunacion v
    JOIN dim_territorio t ON t.territorio_key = v.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = v.fecha_key
    WHERE t.pais = 'Chile' AND v.pct_esquema_completo IS NOT NULL
    ORDER BY d.fecha DESC
    LIMIT 1
"""
chile_vac = pd.read_sql(CHILE_VAC, dw)

control_vac = pd.DataFrame({
    "medida": ["filas", "Chile · fecha del último % esquema completo", "Chile · último % esquema completo"],
    "OWID": [len(v), chile_owid["date"], chile_owid["people_fully_vaccinated_per_hundred"]],
    "DW": [pd.read_sql("SELECT COUNT(*) n FROM fact_vacunacion", dw)["n"][0],
           chile_vac["fecha"][0], chile_vac["pct_esquema_completo"][0]],
})
control_vac["cuadra"] = (control_vac["OWID"] == control_vac["DW"]).map({True: "OK", False: "REVISAR"})
control_vac

,medida,OWID,DW,cuadra
0,filas,54905,54905,OK
1,Chile · fecha del último % esquema completo,2023-02-08,2023-02-08,OK
2,Chile · último % esquema completo,90.29,90.29,OK


## 6 · Fuente externa 2: Banco Mundial (H3.3)
La advertencia de P7 dice que el **ingreso** y la **edad** de la población también influyen en los fallecidos.
El Banco Mundial publica ambos por país; se traen los valores de **2021** (el año antes de la ola de ómicron):

| Indicador | Código | Columna en el DW |
|---|---|---|
| % de la población con 65 años o más | `SP.POP.65UP.TO.ZS` | `pct_65_mas` |
| PIB per cápita (US$ corrientes) | `NY.GDP.PCAP.CD` | `pib_per_capita` |

- La API (`api.worldbank.org/v2`, nombrada en la PPT S14) no entrega CSV sino **JSON**: una lista de dos partes,
  `[metadatos, datos]`. Se lee con `json` y `urllib`, que vienen incluidos en Python.
- Es un valor fijo por país, así que va como **atributo de `dim_territorio`** (igual que `poblacion_pais`): se repite en
  todas las filas del país y no hace falta otra tabla de hechos.
- Se cruza por ISO3, como OWID. Kosovo vuelve a tener otro código: el Banco Mundial usa `XKX`.
- Camas hospitalarias (`SH.MED.BEDS.ZS`) no se trae: solo 124 países tienen el dato de 2021.

In [12]:
import json
import urllib.request

URL_BM = "https://api.worldbank.org/v2/country/all/indicator/{}?format=json&per_page=20000&date=2021"
INDICADORES = {"SP.POP.65UP.TO.ZS": "pct_65_mas", "NY.GDP.PCAP.CD": "pib_per_capita"}

bm = None
for codigo, columna in INDICADORES.items():
    datos = json.load(urllib.request.urlopen(URL_BM.format(codigo)))[1]      # [0] = metadatos, [1] = datos
    tabla = pd.DataFrame({"iso3":    [d["countryiso3code"] for d in datos],
                          "pais_bm": [d["country"]["value"] for d in datos],
                          columna:   [d["value"] for d in datos]})
    bm = tabla if bm is None else bm.merge(tabla, on=["iso3", "pais_bm"])

bm["iso3"] = bm["iso3"].replace({"XKX": "XKS"})          # Kosovo: el Banco Mundial usa XKX y UID usa XKS
print("Banco Mundial:", len(bm), "filas (países y agregados)")
bm.head()

Banco Mundial: 266 filas (países y agregados)


,iso3,pais_bm,pct_65_mas,pib_per_capita
0,AFE,Africa Eastern and Southern,3.216573,1560.894626
1,AFW,Africa Western and Central,2.931946,2116.938294
2,ARB,Arab World,4.339114,6708.207619
3,CSS,Caribbean small states,9.174453,13996.447507
4,CEB,Central Europe and the Baltics,19.493288,19224.328774


### Tasa de cruce

In [13]:
cruce_bm = paises_dw[["pais", "iso3"]].merge(bm, on="iso3", how="left")

for columna in INDICADORES.values():
    print(f"{columna}: {cruce_bm[columna].notna().sum()} de {len(cruce_bm)} países con dato de 2021")

sin_dato = cruce_bm[cruce_bm[list(INDICADORES.values())].isna().any(axis=1)].copy()
sin_dato["motivo"] = "el Banco Mundial no publica el dato de 2021"
sin_dato.loc[sin_dato["pais_bm"].isna(), "motivo"] = "el país no está en el Banco Mundial"
sin_dato.reset_index(drop=True)

pct_65_mas: 194 de 196 países con dato de 2021
pib_per_capita: 189 de 196 países con dato de 2021


,pais,iso3,pais_bm,pct_65_mas,pib_per_capita,motivo
0,Cuba,CUB,Cuba,15.744578,NaN,el Banco Mundial no publica el dato de 2021
1,Eritrea,ERI,Eritrea,4.100612,NaN,el Banco Mundial no publica el dato de 2021
2,Holy See,VAT,NaN,NaN,NaN,el país no está en el Banco Mundial
3,"Korea, North",PRK,"Korea, Dem. People's Rep.",11.282498,NaN,el Banco Mundial no publica el dato de 2021
4,South Sudan,SSD,South Sudan,2.761693,NaN,el Banco Mundial no publica el dato de 2021
5,Taiwan*,TWN,NaN,NaN,NaN,el país no está en el Banco Mundial
6,Yemen,YEM,"Yemen, Rep.",2.518882,NaN,el Banco Mundial no publica el dato de 2021


### Lo que está en el Banco Mundial y no en el DW
Son sobre todo **agregados** (mundo, regiones, grupos de ingreso) y territorios que en JHU van como región de otro país.

In [14]:
otros_bm = bm.loc[~bm["iso3"].isin(paises_dw["iso3"]), ["iso3", "pais_bm"]].copy()
otros_bm["motivo"] = "agregado del Banco Mundial o territorio que no está en JHU"
otros_bm.loc[otros_bm["iso3"].isin(region_de.index), "motivo"] = "en JHU es una región de " + otros_bm["iso3"].map(region_de)

print("Códigos del Banco Mundial sin pareja:", len(otros_bm),
      "| regiones de otro país en JHU:", otros_bm["motivo"].str.startswith("en JHU").sum())
otros_bm.sort_values(["motivo", "iso3"]).reset_index(drop=True)

Códigos del Banco Mundial sin pareja: 72 | regiones de otro país en JHU: 21


,iso3,pais_bm,motivo
0,,High income,agregado del Banco Mundial o territorio que no...
1,,Low income,agregado del Banco Mundial o territorio que no...
2,,Lower middle income,agregado del Banco Mundial o territorio que no...
3,,Not classified,agregado del Banco Mundial o territorio que no...
4,,Upper middle income,agregado del Banco Mundial o territorio que no...
...,...,...,...
67,CYM,Cayman Islands,en JHU es una región de United Kingdom
68,GIB,Gibraltar,en JHU es una región de United Kingdom
69,IMN,Isle of Man,en JHU es una región de United Kingdom
70,TCA,Turks and Caicos Islands,en JHU es una región de United Kingdom


### Agregar a `dim_territorio` y conciliar

In [15]:
filas_antes = len(dim_territorio)
dim_territorio = dim_territorio.merge(cruce_bm[["pais", *INDICADORES.values()]], on="pais", how="left")
dim_territorio.to_sql("dim_territorio", dw, if_exists="replace", index=False)
print("dim_territorio:", len(dim_territorio), "filas (antes:", filas_antes, ") |", dim_territorio.shape[1], "columnas")

chile_bm = bm[bm["iso3"] == "CHL"].iloc[0]
chile_bm_dw = pd.read_sql("SELECT DISTINCT pct_65_mas, pib_per_capita FROM dim_territorio WHERE pais = 'Chile'", dw)

control_bm = pd.DataFrame({
    "medida": ["Chile · % 65 años o más (2021)", "Chile · PIB per cápita (2021)", "Chile · valores distintos en sus 18 filas (debe ser 1)"],
    "Banco Mundial": [chile_bm["pct_65_mas"], chile_bm["pib_per_capita"], 1],
    "DW": [chile_bm_dw["pct_65_mas"][0], chile_bm_dw["pib_per_capita"][0], len(chile_bm_dw)],
})
control_bm["cuadra"] = (control_bm["Banco Mundial"] == control_bm["DW"]).map({True: "OK", False: "REVISAR"})
control_bm

dim_territorio: 4280 filas (antes: 4280 ) | 13 columnas


,medida,Banco Mundial,DW,cuadra
0,Chile · % 65 años o más (2021),12.956434,12.956434,OK
1,Chile · PIB per cápita (2021),16216.184086,16216.184086,OK
2,Chile · valores distintos en sus 18 filas (deb...,1.000000,1.000000,OK


## 7 · Consultas OLAP (H3.4)
Las operaciones de la S9 y la S10, hechas **sobre el DW** con SQL (`pd.read_sql`). Pandas solo se usa para ordenar o
rotar el resultado, que ya es chico.

| Operación | Qué hace | Consulta |
|---|---|---|
| Roll-up | Resumir: subir de nivel en una jerarquía | Casos del mundo: mes → año |
| Drill-down | Detallar: bajar de nivel | Chile 2022: país → regiones |
| Slice | Fijar **una** dimensión | Ola de ómicron (dic-2021 a mar-2022): fallecidos por país |
| Dice | Fijar **varias** dimensiones | 4 países de Sudamérica × 2021 |
| Pivot | Rotar una dimensión a las columnas | Regiones de Chile × año |

**Cuidado al sumar casos nuevos.** El día en que un territorio cambia de granularidad (Chile el 20-may-2020, condados
de EE.UU. el 22-mar-2020) sus casos nuevos quedan vacíos (H2.1). Sumar `casos_nuevos` sirve dentro de un mismo nivel,
pero el total de un país pierde ese día. Para totales exactos de un país se resta el **acumulado** del último día de
cada período (opción C). La sección 7.1 lo muestra con Chile.

### 7.1 · Roll-up: del mes al año

In [16]:
POR_MES = """
    SELECT d.anio, d.mes, SUM(f.casos_nuevos) AS casos, SUM(f.fallecidos_nuevos) AS fallecidos
    FROM fact_covid f
    JOIN dim_fecha d ON d.fecha_key = f.fecha_key
    GROUP BY d.anio, d.mes
    ORDER BY d.anio, d.mes
"""
POR_ANIO = """
    SELECT d.anio, SUM(f.casos_nuevos) AS casos, SUM(f.fallecidos_nuevos) AS fallecidos
    FROM fact_covid f
    JOIN dim_fecha d ON d.fecha_key = f.fecha_key
    GROUP BY d.anio
    ORDER BY d.anio
"""
por_mes = pd.read_sql(POR_MES, dw)
por_anio = pd.read_sql(POR_ANIO, dw)
print("Mundo · casos y fallecidos nuevos por mes (primeros 6 de", len(por_mes), "meses):")
print(por_mes.head(6).to_string(index=False))
print("\nRoll-up al año:")
por_anio

Mundo · casos y fallecidos nuevos por mes (primeros 6 de 39 meses):
 anio  mes     casos  fallecidos
 2020    1    9890.0       207.0
 2020    2   75302.0      2724.0
 2020    3  780063.0     43240.0
 2020    4 2403687.0    205077.0
 2020    5 2865143.0    149484.0
 2020    6 4329919.0    148632.0

Roll-up al año:


,anio,casos,fallecidos
0,2020,83923856.0,1919472.0
1,2021,204957775.0,3567087.0
2,2022,371734079.0,1224611.0
3,2023,16106442.0,188428.0


### Control: Chile con acumulados vs suma de casos nuevos
El total de Chile por mes se calcula de las dos formas. Con acumulados tiene que cerrar en 5.192.286;
con la suma de casos nuevos falta el día en que Chile pasó de nacional a regional.

In [17]:
CHILE_DIA = """
    SELECT d.fecha, d.anio, d.mes,
           SUM(f.casos_acumulados) AS acumulado, SUM(f.casos_nuevos) AS suma_nuevos
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile'
    GROUP BY d.fecha
    ORDER BY d.fecha
"""
chile_dia = pd.read_sql(CHILE_DIA, dw)

chile_mes = chile_dia.groupby(["anio", "mes"]).agg(acumulado=("acumulado", "last"), suma_nuevos=("suma_nuevos", "sum"))
chile_mes["casos_mes"] = chile_mes["acumulado"].diff().fillna(chile_mes["acumulado"])   # último día del mes − el del mes anterior

print("Chile · con acumulados:", int(chile_mes["casos_mes"].sum()),
      "| sumando casos nuevos:", int(chile_mes["suma_nuevos"].sum()),
      "| diferencia:", int(chile_mes["casos_mes"].sum() - chile_mes["suma_nuevos"].sum()))
print("\nMeses en que difieren:")
chile_mes[chile_mes["casos_mes"] != chile_mes["suma_nuevos"]]

Chile · con acumulados: 5192286 | sumando casos nuevos: 5188248 | diferencia: 4038

Meses en que difieren:


,,acumulado,suma_nuevos,casos_mes
anio,mes,,,
2020,5,99688.0,79627.0,83665.0


### 7.2 · Drill-down: Chile 2022, de país a regiones
En 2022 Chile informa siempre por región, así que aquí sí se pueden sumar los casos nuevos.
La tasa por 100.000 usa la población de UID (`poblacion_pais` para el país, `poblacion` para cada región).

In [18]:
CHILE_2022 = """
    SELECT SUM(f.casos_nuevos) AS casos, MAX(t.poblacion_pais) AS poblacion
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile' AND d.anio = 2022
"""
REGIONES_2022 = """
    SELECT t.region_estado AS region, SUM(f.casos_nuevos) AS casos, MAX(t.poblacion) AS poblacion
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile' AND t.nivel = 'region' AND d.anio = 2022
    GROUP BY t.region_estado
"""
pais_2022 = pd.read_sql(CHILE_2022, dw)
regiones_2022 = pd.read_sql(REGIONES_2022, dw)
regiones_2022["por_100k"] = (regiones_2022["casos"] / regiones_2022["poblacion"] * 100_000).round(0)

print("Chile 2022:", int(pais_2022["casos"][0]), "casos |",
      round(pais_2022["casos"][0] / pais_2022["poblacion"][0] * 100_000), "por 100.000")
print("Las regiones suman lo mismo que el país:", regiones_2022["casos"].sum() == pais_2022["casos"][0])
regiones_2022.sort_values("por_100k", ascending=False).reset_index(drop=True)

Chile 2022: 3231033 casos | 16902 por 100.000
Las regiones suman lo mismo que el país: True


,region,casos,poblacion,por_100k
0,Atacama,81017.0,288944.0,28039.0
1,Aysen,24681.0,103158.0,23925.0
2,Nuble,113268.0,480609.0,23568.0
3,Los Rios,89851.0,384837.0,23348.0
4,Maule,228032.0,1044950.0,21822.0
5,Arica y Parinacota,49270.0,226068.0,21794.0
6,Magallanes,36269.0,166533.0,21779.0
7,Biobio,327643.0,1556805.0,21046.0
8,Tarapaca,65850.0,330558.0,19921.0
9,Coquimbo,148886.0,757586.0,19653.0


### 7.3 · Slice: la ola de ómicron
Una sola dimensión fija: la fecha, del 1-dic-2021 al 31-mar-2022 (la ventana de P7). Fallecidos por 100.000 por país,
solo países con 1 millón de habitantes o más (como el ranking de H2.4).

In [19]:
OMICRON = """
    SELECT t.pais, SUM(f.fallecidos_nuevos) AS fallecidos, MAX(t.poblacion_pais) AS poblacion
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE d.fecha BETWEEN '2021-12-01' AND '2022-03-31'
    GROUP BY t.pais
"""
omicron = pd.read_sql(OMICRON, dw)
omicron = omicron[omicron["poblacion"] >= 1_000_000].copy()
omicron["por_100k"] = (omicron["fallecidos"] / omicron["poblacion"] * 100_000).round(1)
omicron = omicron.sort_values("por_100k", ascending=False).reset_index(drop=True)

print("Países:", len(omicron), "| Chile:", omicron.loc[omicron["pais"] == "Chile", "por_100k"].iloc[0],
      "fallecidos por 100.000, puesto", omicron.index[omicron["pais"] == "Chile"][0] + 1)
omicron.head(10)

Países: 157 | Chile: 95.4 fallecidos por 100.000, puesto 6


,pais,fallecidos,poblacion,por_100k
0,Georgia,4641.0,3989175.0,116.3
1,Bulgaria,8076.0,6948445.0,116.2
2,Trinidad and Tobago,1592.0,1399491.0,113.8
3,Croatia,4664.0,4105268.0,113.6
4,Hungary,10949.0,9660350.0,113.3
5,Chile,18234.0,19116209.0,95.4
6,Bosnia and Herzegovina,3127.0,3280815.0,95.3
7,Slovakia,4934.0,5434712.0,90.8
8,Greece,9353.0,10423056.0,89.7
9,Poland,31590.0,37846605.0,83.5


### 7.4 · Dice: cuatro países de Sudamérica en 2021
Varias dimensiones fijas a la vez: país (Chile, Argentina, Perú, Brasil) y año (2021). Casos por 100.000 por mes,
rotados con `pivot_table` para comparar.

In [20]:
SUDAMERICA_2021 = """
    SELECT t.pais, d.mes, SUM(f.casos_nuevos) AS casos, MAX(t.poblacion_pais) AS poblacion
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais IN ('Chile', 'Argentina', 'Peru', 'Brazil') AND d.anio = 2021
    GROUP BY t.pais, d.mes
"""
sud = pd.read_sql(SUDAMERICA_2021, dw)
sud["por_100k"] = (sud["casos"] / sud["poblacion"] * 100_000).round(0)
sud.pivot_table(index="pais", columns="mes", values="por_100k")

mes,1,2,3,4,5,6,7,8,9,10,11,12
pais,,,,,,,,,,,,
Argentina,668.0,399.0,534.0,1391.0,1780.0,1524.0,1016.0,566.0,158.0,71.0,93.0,716.0
Brazil,719.0,635.0,1037.0,899.0,886.0,947.0,636.0,406.0,304.0,180.0,134.0,90.0
Chile,618.0,510.0,894.0,1060.0,974.0,897.0,313.0,120.0,82.0,213.0,354.0,229.0
Peru,373.0,563.0,682.0,760.0,473.0,293.0,180.0,116.0,78.0,80.0,105.0,183.0


### 7.5 · Pivot: regiones de Chile × año
Casos por 100.000 por región (filas) y año (columnas). Ojo: 2020 parte el 20-may, cuando empiezan los datos regionales
(y ese primer día queda vacío), y 2023 llega solo al 9-mar.

In [21]:
REGIONES_ANIO = """
    SELECT t.region_estado AS region, d.anio, SUM(f.casos_nuevos) AS casos, MAX(t.poblacion) AS poblacion
    FROM fact_covid f
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    WHERE t.pais = 'Chile' AND t.nivel = 'region'
    GROUP BY t.region_estado, d.anio
"""
reg = pd.read_sql(REGIONES_ANIO, dw)
reg["por_100k"] = (reg["casos"] / reg["poblacion"] * 100_000).round(0)
reg.pivot_table(index="region", columns="anio", values="por_100k").sort_values(2022, ascending=False)

anio,2020,2021,2022,2023
region,,,,
Atacama,2886.0,7725.0,28039.0,1770.0
Aysen,1518.0,8470.0,23925.0,1318.0
Nuble,2018.0,5991.0,23568.0,1162.0
Los Rios,2236.0,11690.0,23348.0,1185.0
Maule,2372.0,8068.0,21822.0,861.0
Arica y Parinacota,4592.0,8033.0,21794.0,656.0
Magallanes,9543.0,8001.0,21779.0,715.0
Biobio,2965.0,7732.0,21046.0,984.0
Tarapaca,4415.0,8904.0,19921.0,764.0


## 8 · Guardar comprimido para GitHub
El `.db` pesa unos 130 MB y GitHub no acepta archivos de más de 100 MB. Por eso al repositorio se sube
`data/dw/covid_dw.zip` (unos 47 MB) y el `.db` queda fuera (`.gitignore`). `zipfile` viene incluido en Python.

Para usar el DW en otro notebook o en el tablero, primero se descomprime:
`zipfile.ZipFile("../data/dw/covid_dw.zip").extractall("../data/dw")`.

In [22]:
import zipfile

dw.close()
RUTA_ZIP = "../data/dw/covid_dw.zip"
with zipfile.ZipFile(RUTA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RUTA_DW, "covid_dw.db")

print(f"covid_dw.db: {os.path.getsize(RUTA_DW) / 2**20:.1f} MB → covid_dw.zip: {os.path.getsize(RUTA_ZIP) / 2**20:.1f} MB")

covid_dw.db: 131.5 MB → covid_dw.zip: 45.7 MB
